# Testando Cliente MCP com Servidor MCP TypeScript no Amazon Bedrock AgentCore Runtime

## Visão Geral

Neste tutorial, aprenderemos como hospedar um servidor MCP (Model Context Protocol) baseado em TypeScript usando o ambiente de runtime do Amazon Bedrock AgentCore.

### Detalhes do Tutorial

| Informação          | Detalhes                                                  |
|:--------------------|:----------------------------------------------------------|
| Tipo de tutorial    | Hospedagem de servidor MCP em TypeScript                  |
| Tipo de ferramenta  | Servidor MCP                                              |
| Componentes         | Hospedagem de servidor MCP em TypeScript no AgentCore Runtime |
| Vertical            | Cross-vertical                                            |
| Complexidade        | Fácil                                                     |
| SDK utilizado       | SDK TypeScript da Anthropic para MCP                      |


### Visão Geral do Tutorial

1. A autenticação do AgentCore Runtime usará o Amazon Cognito para fornecer tokens JWT para acessar nosso servidor MCP implantado.

2. O servidor MCP é escrito em TypeScript e será [implantado usando fluxo customizado](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/getting-started-custom.html)

3. O cliente MCP é escrito em Python.
   _Nota: o cliente MCP pode ser escrito em qualquer linguagem._

## Pré-requisitos

Para executar este tutorial você precisará de:
- Node.js v22 ou superior (Servidor MCP)
- Python 3.10+ (Cliente MCP)
- Docker (para containerização)  
- Amazon ECR (Elastic Container Registry) para armazenar imagens Docker  
- Conta AWS com acesso ao Bedrock AgentCore  
- Biblioteca MCP (Model Context Protocol)
- Docker em execução

In [ ]:
#!uv add -r requirements.txt --active

## Entendendo o MCP (Model Context Protocol)

MCP é um protocolo que permite que modelos de IA acessem dados e ferramentas externas de forma segura. Conceitos principais:

* **Tools**: Funções que a IA pode chamar para executar ações
* **Prompts**: Prompts permitem que servidores forneçam mensagens e instruções estruturadas para interagir com LLMs
* **Streamable HTTP**: Protocolo de transporte usado pelo AgentCore Runtime
* **Isolamento de Sessão**: Cada cliente recebe sessões isoladas via cabeçalho `Mcp-Session-Id`
* **Operação Stateless**: Servidores devem suportar operação stateless para escalabilidade

O AgentCore Runtime espera que os servidores MCP sejam hospedados em `0.0.0.0:8000/mcp` como o caminho padrão.

## Passo 1: Configurando o Amazon Cognito para Autenticação

O AgentCore Runtime requer autenticação. Usaremos o Amazon Cognito para fornecer tokens JWT para acessar nosso servidor MCP implantado.

In [ ]:
import sys
import os

# Get the current notebook's directory
current_dir = os.path.dirname(os.path.abspath('__file__' if '__file__' in globals() else '.'))

utils_dir = os.path.join(current_dir, '..')
utils_dir = os.path.abspath(utils_dir)

# Add to sys.path
sys.path.insert(0, utils_dir)
print("sys.path[0]:", sys.path[0])

from utils import create_agentcore_role, setup_cognito_user_pool

In [ ]:
print("Configurando o user pool do Amazon Cognito...")
cognito_config = setup_cognito_user_pool()
print("Configuração do Cognito concluída ✓")
print(f"User Pool ID: {cognito_config.get('user_pool_id', 'N/A')}")
print(f"Client ID: {cognito_config.get('client_id', 'N/A')}")

## Passo 2: Criar Role de Execução IAM

Antes de começar, vamos criar uma role IAM para nosso AgentCore Runtime. Esta role fornece as permissões necessárias para o runtime operar.

In [ ]:
tool_name = "mcp_server_ac"
print(f"Criando role IAM para {tool_name}...")
agentcore_iam_role = create_agentcore_role(agent_name=tool_name)
print(f"Role IAM criada ✓")
print(f"Role ARN: {agentcore_iam_role['Role']['Arn']}")

## Passo 3: Criando o Servidor MCP

Agora vamos criar nosso servidor MCP em TypeScript com duas ferramentas simples e um prompt. Navegue até a pasta src neste tutorial.

1. Instalar dependências

```
npm install
```

2. Configurar credenciais AWS
```
aws configure
export AWS_ACCESS_KEY_ID=your_access_key
export AWS_SECRET_ACCESS_KEY=your_secret_key
export AWS_REGION=us-east-1
```

3. Iniciar servidor (executando localmente)
```
npm run start
```

## Passo 4: Implantação do Servidor MCP via Docker

Nota: Estes são passos manuais para implantar um agent ou servidor MCP sem o starter toolkit 

https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/getting-started-custom.html

1. Criar Repositório ECR
```
aws ecr create-repository --repository-name mcp-server --region us-east-1
```
2. Build e Push da Imagem para ECR
```
# Obter token de login
aws ecr get-login-password --region us-east-1 | \
  docker login --username AWS --password-stdin [account-id].dkr.ecr.us-east-1.amazonaws.com

docker buildx --platform linux/arm64 \
  -t [account-id].dkr.ecr.us-east-1.amazonaws.com/mcp-server:latest --push .
```

3. Implantar no Bedrock AgentCore

    - Vá para AWS Console → Bedrock → AgentCore → Create Agent
    - Escolha MCP como protocolo
    - Configure o Agent Runtime:
        - Image URI: [account-id].dkr.ecr.us-east-1.amazonaws.com/mcp-server:latest
        - Configure as permissões IAM para acesso ao modelo Bedrock
        - Implante e teste no Agent Sandbox
    - Para Discovery url: Selecione a URL acima, cognito_config['discovery_url']
    - Para Client id: Selecione o client id acima, cognito_config['client_id']
    - Para execution role: Selecione o ARN acima, agentcore_iam_role['Role']['Arn']


## Passo 5: Armazenando a Configuração para Acesso Remoto

Antes de invocar nosso servidor MCP implantado, vamos armazenar o Agent ARN (obtenha o ARN do Passo 4) e a configuração do Cognito no AWS Systems Manager Parameter Store e AWS Secrets Manager para recuperação fácil:

In [ ]:
import boto3
import json

boto_session = Session()
region = boto_session.region_name

ssm_client = boto3.client('ssm', region_name=region)
secrets_client = boto3.client('secretsmanager', region_name=region)

try:
    cognito_credentials_response = secrets_client.create_secret(
        Name='mcp_server/cognito/credentials',
        Description='Cognito credentials for MCP server',
        SecretString=json.dumps(cognito_config)
    )
    print("✓ Credenciais do Cognito armazenadas no Secrets Manager")
except secrets_client.exceptions.ResourceExistsException:
    secrets_client.update_secret(
        SecretId='mcp_server/cognito/credentials',
        SecretString=json.dumps(cognito_config)
    )
    print("✓ Credenciais do Cognito atualizadas no Secrets Manager")

 # NOTA: Adicione o ARN do seu agent que você criou no Passo 4
agent_arn_response = ssm_client.put_parameter(
    Name='/mcp_server/runtime/agent_arn',
    Value="Adicione o ARN do seu agent que você criou no passo 4", 
    Type='String',
    Description='Agent ARN for MCP server',
    Overwrite=True
)
print("✓ Agent ARN armazenado no Parameter Store")

## Passo 6: Criando o Cliente de Teste Remoto

Agora vamos criar um cliente para testar nosso servidor MCP implantado. Este cliente recuperará as credenciais necessárias da AWS e se conectará ao servidor implantado:

In [ ]:
%%writefile my_mcp_client_remote.py
import asyncio
import boto3
import json
import sys
from boto3.session import Session

from mcp import ClientSession
from mcp.client.streamable_http import streamablehttp_client

async def main():
    boto_session = Session()
    region = boto_session.region_name
    
    print(f"Usando região AWS: {region}")
    
    try:
        ssm_client = boto3.client('ssm', region_name=region)
        agent_arn_response = ssm_client.get_parameter(Name='/mcp_server/runtime/agent_arn')
        agent_arn = agent_arn_response['Parameter']['Value']
        print(f"Agent ARN recuperado: {agent_arn}")
     
        secrets_client = boto3.client('secretsmanager', region_name=region)
        response = secrets_client.get_secret_value(SecretId='mcp_server/cognito/credentials')
        secret_value = response['SecretString']
        parsed_secret = json.loads(secret_value)
        bearer_token = parsed_secret['bearer_token']
        print("✓ Bearer token recuperado do Secrets Manager")
        
    except Exception as e:
        print(f"Erro ao recuperar credenciais: {e}")
        sys.exit(1)
    
    if not agent_arn or not bearer_token:
        print("Erro: BEARER_TOKEN não foi recuperado corretamente")
        sys.exit(1)
    

    encoded_arn = agent_arn.replace(':', '%3A').replace('/', '%2F')
    mcp_url = f"https://bedrock-agentcore.{region}.amazonaws.com/runtimes/{encoded_arn}/invocations?qualifier=DEFAULT"
    headers = {
        "authorization": f"Bearer {bearer_token}",
        "Content-Type": "application/json"
    }
    
    print(f"\nConectando a: {mcp_url}")
    print("Headers configurados ✓")

    try:
        async with streamablehttp_client(mcp_url, headers, timeout=120, terminate_on_close=False) as (
            read_stream,
            write_stream,
            _,
        ):
            async with ClientSession(read_stream, write_stream) as session:
                print("\n🔄 Inicializando sessão MCP...")
                await session.initialize()
                print("✓ Sessão MCP inicializada")
                
                print("\n🔄 Listando ferramentas disponíveis...")
                tool_result = await session.list_tools()
                
                print("\n📋 Ferramentas MCP Disponíveis:")
                print("=" * 50)
                for tool in tool_result.tools:
                    print(f"🔧 {tool.name}")
                    print(f"   Descrição: {tool.description}")
                    if hasattr(tool, 'inputSchema') and tool.inputSchema:
                        properties = tool.inputSchema.get('properties', {})
                        if properties:
                            print(f"   Parâmetros: {list(properties.keys())}")
                    print()
                
                print(f"✅ Conectado com sucesso ao servidor MCP!")
                print(f"Encontradas {len(tool_result.tools)} ferramentas disponíveis.")
                
    except Exception as e:
        print(f"❌ Erro ao conectar ao servidor MCP: {e}")
        sys.exit(1)

if __name__ == "__main__":
    asyncio.run(main())

## Passo 7: Testando seu Servidor MCP Implantado

Vamos testar nosso servidor MCP implantado usando o cliente remoto:

In [ ]:
print("Testando servidor MCP implantado...")
print("=" * 50)
!python my_mcp_client_remote.py

## Passo 8: Invocando Ferramentas MCP Remotamente

Agora vamos criar um cliente aprimorado que não apenas lista as ferramentas, mas também as invoca para demonstrar a funcionalidade completa do MCP:

In [ ]:
%%writefile invoke_mcp_tools.py
import asyncio
import boto3
import json
import sys
from boto3.session import Session

from mcp import ClientSession
from mcp.client.streamable_http import streamablehttp_client

async def main():
    boto_session = Session()
    region = boto_session.region_name
    
    print(f"Usando região AWS: {region}")
    
    try:
        ssm_client = boto3.client('ssm', region_name=region)
        agent_arn_response = ssm_client.get_parameter(Name='/mcp_server/runtime/agent_arn')
        agent_arn = agent_arn_response['Parameter']['Value']
        print(f"Agent ARN recuperado: {agent_arn}")

        secrets_client = boto3.client('secretsmanager', region_name=region)
        response = secrets_client.get_secret_value(SecretId='mcp_server/cognito/credentials')
        secret_value = response['SecretString']
        parsed_secret = json.loads(secret_value)
        bearer_token = parsed_secret['bearer_token']
        print("✓ Bearer token recuperado do Secrets Manager")
        
    except Exception as e:
        print(f"Erro ao recuperar credenciais: {e}")
        sys.exit(1)
    
    encoded_arn = agent_arn.replace(':', '%3A').replace('/', '%2F')
    mcp_url = f"https://bedrock-agentcore.{region}.amazonaws.com/runtimes/{encoded_arn}/invocations?qualifier=DEFAULT"
    headers = {
        "authorization": f"Bearer {bearer_token}",
        "Content-Type": "application/json"
    }
    
    print(f"\nConectando a: {mcp_url}")

    try:
        async with streamablehttp_client(mcp_url, headers, timeout=120, terminate_on_close=False) as (
            read_stream,
            write_stream,
            _,
        ):
            async with ClientSession(read_stream, write_stream) as session:
                print("\n🔄 Inicializando sessão MCP...")
                await session.initialize()
                print("✓ Sessão MCP inicializada")
                
                print("\n🔄 Listando ferramentas disponíveis...")
                tool_result = await session.list_tools()
                
                print("\n📋 Ferramentas MCP Disponíveis:")
                print("=" * 50)
                for tool in tool_result.tools:
                    print(f"🔧 {tool.name}: {tool.description}")
                
                print("\n🧪 Testando Ferramentas MCP:")
                print("=" * 50)
                
                try:
                    print("\n➕ Testando add(5, 3)...")
                    add_result = await session.call_tool(
                        name="add",
                        arguments={"a": 5, "b": 3}
                    )
                    print(f"   Resultado: {add_result.content[0].text}")
                except Exception as e:
                    print(f"   Erro: {e}")
                
                try:
                    print("\n✖️  Testando subtract(10, 2)...")
                    substract_result = await session.call_tool(
                        name="subtract",
                        arguments={"a": 10, "b": 2}
                    )
                    print(f"   Resultado: {substract_result.content[0].text}")
                except Exception as e:
                    print(f"   Erro: {e}")
                
                print("\n✅ Teste de ferramentas MCP concluído!")
                
    except Exception as e:
        print(f"❌ Erro ao conectar ao servidor MCP: {e}")
        sys.exit(1)

if __name__ == "__main__":
    asyncio.run(main())

## Testar Invocação de Ferramentas

Vamos testar nossas ferramentas MCP invocando-as de fato:

In [ ]:
print("Testando invocação de ferramentas MCP...")
print("=" * 50)
!python invoke_mcp_tools.py

## Próximos Passos

Agora que você implantou com sucesso um servidor MCP no AgentCore Runtime, você pode:

1. **Adicionar Mais Ferramentas**: Estenda seu servidor MCP com ferramentas adicionais
2. **Autenticação Customizada**: Implemente autorizadores JWT customizados
3. **Integração**: Integre com outros serviços do AgentCore

# 🎉 Parabéns!

Você completou com sucesso:

✅ **Criou um servidor MCP em TypeScript** com ferramentas customizadas  
✅ **Configurou autenticação** com Amazon Cognito  
✅ **Implantou na AWS** usando AgentCore Runtime  
✅ **Invocou remotamente** com autenticação adequada  
✅ **Aprendeu conceitos de MCP** e melhores práticas  

Seu servidor MCP agora está rodando no Amazon Bedrock AgentCore Runtime e pronto para uso em produção!